In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile

zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/parkinson_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [3]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import os

In [4]:
import os
import shutil

source = "/content/parkinson_dataset/New hand pd Dataset"

target = "/content/parkinson_clean"

healthy_dir = os.path.join(target, "Healthy")
patient_dir = os.path.join(target, "Parkinson")

os.makedirs(healthy_dir, exist_ok=True)
os.makedirs(patient_dir, exist_ok=True)

# Healthy folders
healthy_paths = [
    "Healthy/HealthyCircle",
    "Healthy/HealthyMeander",
    "Healthy/HealthySpiral"
]

# Patient folders
patient_paths = [
    "Patient/PatientCircle",
    "Patient/PatientMeander",
    "Patient/PatientSpiral"
]

# Move Healthy images
for path in healthy_paths:
    full_path = os.path.join(source, path)
    for root, dirs, files in os.walk(full_path):
        for file in files:
            shutil.copy(os.path.join(root, file), healthy_dir)

# Move Patient images
for path in patient_paths:
    full_path = os.path.join(source, path)
    for root, dirs, files in os.walk(full_path):
        for file in files:
            shutil.copy(os.path.join(root, file), patient_dir)

In [5]:
import os

healthy_count = len(os.listdir("/content/parkinson_clean/Healthy"))
patient_count = len(os.listdir("/content/parkinson_clean/Parkinson"))

print("Healthy:", healthy_count)
print("Parkinson:", patient_count)

Healthy: 0
Parkinson: 0


In [6]:
import os

base_path = "/content/parkinson_dataset"

for root, dirs, files in os.walk(base_path):
    print(root)
    break

/content/parkinson_dataset


In [7]:
os.listdir("/content/parkinson_dataset")

["Parkinson's Dataset Final"]

In [8]:
import os

for root, dirs, files in os.walk("/content/parkinson_dataset"):
    print(root)
    break

/content/parkinson_dataset


In [9]:
os.listdir("/content/parkinson_dataset")

["Parkinson's Dataset Final"]

In [10]:
import os

os.listdir("/content/parkinson_dataset/Parkinson's Dataset Final")

['New hand pd Dataset', 'Hand Pd']

In [12]:
os.listdir("/content/parkinson_dataset/Parkinson's Dataset Final/New hand pd Dataset")

['Healthy', "Parkinson's"]

In [13]:
import os
import shutil

source = "/content/parkinson_dataset/Parkinson's Dataset Final/New hand pd Dataset"

target = "/content/parkinson_clean"

healthy_src = os.path.join(source, "Healthy")
patient_src = os.path.join(source, "Parkinson's")

healthy_dst = os.path.join(target, "Healthy")
patient_dst = os.path.join(target, "Parkinson")

os.makedirs(healthy_dst, exist_ok=True)
os.makedirs(patient_dst, exist_ok=True)

In [14]:
for root, dirs, files in os.walk(healthy_src):
    for file in files:
        if file.lower().endswith((".png",".jpg",".jpeg")):
            shutil.copy(os.path.join(root,file), healthy_dst)

for root, dirs, files in os.walk(patient_src):
    for file in files:
        if file.lower().endswith((".png",".jpg",".jpeg")):
            shutil.copy(os.path.join(root,file), patient_dst)

In [15]:
print("Healthy:", len(os.listdir("/content/parkinson_clean/Healthy")))
print("Parkinson:", len(os.listdir("/content/parkinson_clean/Parkinson")))

Healthy: 233
Parkinson: 279


In [16]:
import tensorflow as tf
from tensorflow.keras import layers

In [17]:
IMG_SIZE = 224
BATCH_SIZE = 16
SEED = 123

In [18]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/parkinson_clean",
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/parkinson_clean",
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

Found 512 files belonging to 2 classes.
Using 410 files for training.
Found 512 files belonging to 2 classes.
Using 102 files for validation.


In [19]:
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

In [20]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [21]:
train_ds = train_ds.map(lambda x, y: (data_augmentation(x), y))

In [22]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [23]:
for images, labels in train_ds.take(1):
    print(images.shape)
    print(labels.shape)

(16, 224, 224, 3)
(16,)


In [24]:
class PatchEmbedding(layers.Layer):

    def __init__(self, patch_size=16, embed_dim=512):
        super().__init__()

        self.projection = layers.Conv2D(
            filters=embed_dim,
            kernel_size=patch_size,
            strides=patch_size,
            padding="valid"
        )

        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, x):

        x = self.projection(x)
        x = self.flatten(x)

        return x

In [25]:
def transformer_block(x, embed_dim):

    attention = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=embed_dim
    )(x, x)

    x = layers.Add()([x, attention])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(embed_dim * 2, activation="relu")(x)
    ffn = layers.Dense(embed_dim)(ffn)

    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    return x

In [26]:
inputs = layers.Input(shape=(224,224,3))

# Patch embedding
x = PatchEmbedding(16,512)(inputs)

# Vision Transformer layers
x = transformer_block(x,512)

x = layers.Dense(256)(x)
x = transformer_block(x,256)

x = layers.Dense(128)(x)
x = transformer_block(x,128)

x = layers.Dense(64)(x)
x = transformer_block(x,64)

# Global pooling
x = layers.GlobalAveragePooling1D()(x)

# FCNN classifier
x = layers.Dense(32,activation="relu")(x)
x = layers.Dense(16,activation="relu")(x)
x = layers.Dense(8,activation="relu")(x)
x = layers.Dense(4,activation="relu")(x)

outputs = layers.Dense(2,activation="softmax")(x)

model = tf.keras.Model(inputs,outputs)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embedding     │ (None, 196, 512)  │    393,728 │ input_layer_1[0]… │
│ (PatchEmbedding)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 196, 512)  │  4,200,960 │ patch_embedding[… │
│ (MultiHeadAttentio… │                   │            │ patch_embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 196, 512)  │          0 │ patch_embedding[… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 196, 512)  │      1,024 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 196, 1024) │    525,312 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 196, 512)  │    524,800 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 196, 512)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 512)  │      1,024 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 196, 256)  │    131,328 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 196, 256)  │  1,051,904 │ dense_2[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 196, 256)  │          0 │ dense_2[0][0],    │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 196, 512)  │    131,584 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 196, 256)  │    131,328 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 196, 256)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 196, 128)  │     32,896 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 196, 128)  │    263,808 │ dense_5[0][0],  

 Total params: 7,551,398 (28.81 MB)

 Trainable params: 7,551,398 (28.81 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

In [28]:
history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=40
)

Epoch 1/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 49s 848ms/step - accuracy: 0.5209 - loss: 0.9488 - val_accuracy: 0.5392 - val_loss: 0.6931
Epoch 2/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.5613 - loss: 0.6930 - val_accuracy: 0.5392 - val_loss: 0.6930
Epoch 3/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.5381 - loss: 0.6930 - val_accuracy: 0.5392 - val_loss: 0.6930
Epoch 4/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step - accuracy: 0.5479 - loss: 0.6929 - val_accuracy: 0.5392 - val_loss: 0.6929
Epoch 5/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.5382 - loss: 0.6929 - val_accuracy: 0.5392 - val_loss: 0.6929
Epoch 6/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.5692 - loss: 0.6926 - val_accuracy: 0.5392 - val_loss: 0.6928
Epoch 7/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.5695 - loss: 0.6925 - val_accuracy: 0.5392 - val_loss: 0.6927
Epoch 8/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.5470 - loss: 0.6926 - val_accuracy: 0.5392 -